In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
from typing import Dict, List, Tuple, Optional
import time
from tqdm import tqdm
import os
import argparse
import h5py
import random

In [ ]:
class TrajectoryDataset(Dataset):
    """Dataset for neural ODE training from HDF5 files."""

    def __init__(self,
                 hdf5_file: str,
                 operator_type: str,
                 split: str = 'train'):
        """
        Initialize trajectory dataset from HDF5 file.
        
        Args:
            hdf5_file: Path to HDF5 file
            operator_type: 'heat' or 'dispersion'
            split: 'train', 'valid', or 'test'
        """
        self.hdf5_file = hdf5_file
        self.operator_type = operator_type
        self.split = split

        # Load data from HDF5 file
        with h5py.File(hdf5_file, 'r') as f:
            print(f"Loading {operator_type} data from {hdf5_file}")
            print(f"Available groups: {list(f.keys())}")

            # Load trajectories - assuming structure similar to train_combined.py
            if split in f:
                group = f[split]
                if 'pde_250-256' in group:
                    self.trajectories = group['pde_250-256'][:]  # (n_samples, n_timesteps, n_spatial)
                    self.alpha = group['alpha'][:]
                    self.beta = group['beta'][:]
                    self.gamma = group['gamma'][:]
                else:
                    raise ValueError(f"Expected 'pde_250-256' not found in group {split}")
            else:
                raise ValueError(f"Split '{split}' not found in file {hdf5_file}")

        self.n_samples, self.n_timesteps, self.n_spatial = self.trajectories.shape
        print(f"Loaded {self.n_samples} trajectories with shape ({self.n_timesteps}, {self.n_spatial})")

        # Create time points (assuming uniform spacing)
        self.time_points = np.linspace(0, 4.0, self.n_timesteps)
        self.dt = self.time_points[1] - self.time_points[0]

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        # Get appropriate parameter based on operator type
        if self.operator_type == 'heat':
            param_value = self.gamma[idx]  # gamma is typically the diffusion coefficient
        elif self.operator_type == 'dispersion':
            param_value = self.beta[idx]   # beta is typically the advection coefficient
        else:
            param_value = self.alpha[idx]  # fallback

        return {
            'u_sequence': torch.from_numpy(self.trajectories[idx]).float(),
            't_sequence': torch.from_numpy(self.time_points).float(),
            'parameter': torch.tensor(param_value).float(),
            'traj_idx': idx}

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy.integrate import trapz, simpson
from typing import Dict, List, Tuple, Optional
import warnings

class KdVValidator:
    """
    Comprehensive validation suite for KdV equation solutions.
    Tests conservation laws, residuals, and analytical solutions.
    """
    
    def __init__(self, dataset, domain_length: float = None):
        """
        Initialize validator with trajectory dataset.
        
        Args:
            dataset: TrajectoryDataset instance containing KdV solutions
            domain_length: Length of spatial domain (default: auto-detect or use 2π)
        """
        self.dataset = dataset
        self.n_samples = dataset.n_samples
        self.n_timesteps = dataset.n_timesteps
        self.n_spatial = dataset.n_spatial
        self.time_points = dataset.time_points
        self.dt = dataset.dt
        
        # Set spatial domain parameters
        if domain_length is not None:
            self.L = domain_length
        else:
            # Try to auto-detect from dataset attributes, otherwise default to 2π
            if hasattr(dataset, 'domain_length'):
                self.L = dataset.domain_length
            elif hasattr(dataset, 'L'):
                self.L = dataset.L
            else:
                self.L = 2 * np.pi
                print(f"Warning: Domain length not specified. Using L = {self.L:.2f}")
        
        self.dx = self.L / self.n_spatial
        self.x = np.linspace(0, self.L - self.dx, self.n_spatial)
        
        print(f"Spatial domain: [0, {self.L:.2f}] with dx = {self.dx:.4f}")
        print(f"Temporal domain: [0, {self.time_points[-1]:.2f}] with dt = {self.dt:.4f}")
    
    def compute_spatial_derivatives(self, u: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        """
        Compute spatial derivatives using finite differences with periodic BCs.
        
        Args:
            u: Solution array of shape (n_timesteps, n_spatial)
            
        Returns:
            u_x: First derivative
            u_xxx: Third derivative
        """
        # Use FFT for accurate derivatives with periodic BCs
        k = np.fft.fftfreq(self.n_spatial, d=self.dx) * 2 * np.pi
        
        u_x = np.zeros_like(u)
        u_xxx = np.zeros_like(u)
        
        for t in range(u.shape[0]):
            u_hat = np.fft.fft(u[t])
            u_x[t] = np.real(np.fft.ifft(1j * k * u_hat))
            u_xxx[t] = np.real(np.fft.ifft(-1j * k**3 * u_hat))
        
        return u_x, u_xxx
    
    def compute_temporal_derivative(self, u: np.ndarray) -> np.ndarray:
        """
        Compute temporal derivative using finite differences.
        
        Args:
            u: Solution array of shape (n_timesteps, n_spatial)
            
        Returns:
            u_t: Temporal derivative
        """
        u_t = np.zeros_like(u)
        
        # Forward difference at t=0
        u_t[0] = (u[1] - u[0]) / self.dt
        
        # Central difference for interior points
        for t in range(1, u.shape[0] - 1):
            u_t[t] = (u[t+1] - u[t-1]) / (2 * self.dt)
        
        # Backward difference at final time
        u_t[-1] = (u[-1] - u[-2]) / self.dt
        
        return u_t
    
    def check_conservation_laws(self, traj_idx: int = 0, plot: bool = True) -> Dict[str, np.ndarray]:
        """
        Check KdV conservation laws for velocity field u.
        
        For KdV equation u_t + u*u_x + u_xxx = 0, the conserved quantities are:
        - I1 (first invariant): ∫ u dx  
        - I2 (second invariant): ∫ u² dx
        - I3 (third invariant/Hamiltonian): ∫ (u³/3 - (u_x)²/2) dx
        
        Args:
            traj_idx: Index of trajectory to analyze
            plot: Whether to plot conservation quantities
            
        Returns:
            Dictionary containing conservation quantities over time
        """
        data = self.dataset[traj_idx]
        u = data['u_sequence'].numpy()  # (n_timesteps, n_spatial) - velocity field
        
        # Compute derivatives
        u_x, u_xxx = self.compute_spatial_derivatives(u)
        
        # KdV Conservation quantities (invariants)
        I1 = np.zeros(self.n_timesteps)  # First invariant
        I2 = np.zeros(self.n_timesteps)  # Second invariant  
        I3 = np.zeros(self.n_timesteps)  # Third invariant (Hamiltonian)
        
        for t in range(self.n_timesteps):
            # First invariant: ∫ u dx
            I1[t] = trapz(u[t], dx=self.dx)
            
            # Second invariant: ∫ u² dx
            I2[t] = trapz(u[t]**2, dx=self.dx)
            
            # Third invariant (Hamiltonian): ∫ (u³/3 - (u_x)²/2) dx
            I3[t] = trapz(u[t]**3/3 - u_x[t]**2/2, dx=self.dx)
        
        # Compute relative changes from initial values
        I1_change = np.abs(I1 - I1[0]) / (np.abs(I1[0]) + 1e-15)  # Add small epsilon to avoid division by zero
        I2_change = np.abs(I2 - I2[0]) / (np.abs(I2[0]) + 1e-15)
        I3_change = np.abs(I3 - I3[0]) / (np.abs(I3[0]) + 1e-15)
        
        # Debug: Check for problematic values
        if np.any(np.isnan(I1_change)) or np.any(np.isinf(I1_change)):
            print(f"Warning: I1_change contains NaN/Inf. I1[0]={I1[0]}, I1 range=[{np.min(I1)}, {np.max(I1)}]")
        if np.any(np.isnan(I2_change)) or np.any(np.isinf(I2_change)):
            print(f"Warning: I2_change contains NaN/Inf. I2[0]={I2[0]}, I2 range=[{np.min(I2)}, {np.max(I2)}]")
        if np.any(np.isnan(I3_change)) or np.any(np.isinf(I3_change)):
            print(f"Warning: I3_change contains NaN/Inf. I3[0]={I3[0]}, I3 range=[{np.min(I3)}, {np.max(I3)}]")
        
        if plot:
            fig, axes = plt.subplots(2, 2, figsize=(12, 8))
            
            # Plot conservation quantities
            axes[0,0].plot(self.time_points, I1)
            axes[0,0].set_title('First Invariant I₁')
            axes[0,0].set_xlabel('Time')
            axes[0,0].set_ylabel('∫ u dx')
            axes[0,0].grid(True)
            
            axes[0,1].plot(self.time_points, I2)
            axes[0,1].set_title('Second Invariant I₂')
            axes[0,1].set_xlabel('Time')
            axes[0,1].set_ylabel('∫ u² dx')
            axes[0,1].grid(True)
            
            axes[1,0].plot(self.time_points, I3)
            axes[1,0].set_title('Third Invariant I₃ (Hamiltonian)')
            axes[1,0].set_xlabel('Time')
            axes[1,0].set_ylabel('∫ (u³/3 - u_x²/2) dx')
            axes[1,0].grid(True)
            
            # Plot relative changes
            axes[1,1].semilogy(self.time_points, I1_change, label='I₁', linewidth=2)
            axes[1,1].semilogy(self.time_points, I2_change, label='I₂', linewidth=2)
            axes[1,1].semilogy(self.time_points, I3_change, label='I₃', linewidth=2)
            axes[1,1].set_title('Relative Conservation Errors')
            axes[1,1].set_xlabel('Time')
            axes[1,1].set_ylabel('Relative Error')
            axes[1,1].legend()
            axes[1,1].grid(True)
            axes[1,1].set_ylim(1e-16, 1e2)  # Set reasonable y-axis limits
            
            plt.tight_layout()
            plt.show()
        
        return {
            'I1': I1,           # First invariant
            'I2': I2,           # Second invariant
            'I3': I3,           # Third invariant
            'I1_error': I1_change,
            'I2_error': I2_change,
            'I3_error': I3_change
        }
    
    def check_pde_residual(self, traj_idx: int = 0, plot: bool = True) -> Dict[str, np.ndarray]:
        """
        Check PDE residual: |u_t + u*u_x + u_xxx|
        
        Args:
            traj_idx: Index of trajectory to analyze
            plot: Whether to plot residual
            
        Returns:
            Dictionary containing residual information
        """
        data = self.dataset[traj_idx]
        u = data['u_sequence'].numpy()
        
        # Compute all required derivatives
        u_t = self.compute_temporal_derivative(u)
        u_x, u_xxx = self.compute_spatial_derivatives(u)
        
        # Compute PDE residual: u_t + u*u_x + u_xxx = 0
        residual = u_t + u * u_x + u_xxx
        
        # Compute statistics
        residual_rms = np.sqrt(np.mean(residual**2, axis=1))  # RMS over space at each time
        residual_max = np.max(np.abs(residual), axis=1)       # Max over space at each time
        residual_global_rms = np.sqrt(np.mean(residual**2))   # Global RMS
        
        if plot:
            fig, axes = plt.subplots(2, 2, figsize=(12, 8))
            
            # Residual heatmap
            im = axes[0,0].imshow(residual.T, aspect='auto', origin='lower', 
                                 extent=[self.time_points[0], self.time_points[-1], 
                                        self.x[0], self.x[-1]])
            axes[0,0].set_title('PDE Residual: u_t + u*u_x + u_xxx')
            axes[0,0].set_xlabel('Time')
            axes[0,0].set_ylabel('Space')
            plt.colorbar(im, ax=axes[0,0])
            
            # RMS residual over time
            axes[0,1].semilogy(self.time_points, residual_rms)
            axes[0,1].set_title('RMS Residual over Time')
            axes[0,1].set_xlabel('Time')
            axes[0,1].set_ylabel('RMS(residual)')
            axes[0,1].grid(True)
            
            # Max residual over time
            axes[1,0].semilogy(self.time_points, residual_max)
            axes[1,0].set_title('Max Residual over Time')
            axes[1,0].set_xlabel('Time')
            axes[1,0].set_ylabel('Max(|residual|)')
            axes[1,0].grid(True)
            
            # Residual histogram
            axes[1,1].hist(residual.flatten(), bins=50, alpha=0.7)
            axes[1,1].set_title(f'Residual Distribution (Global RMS: {residual_global_rms:.2e})')
            axes[1,1].set_xlabel('Residual Value')
            axes[1,1].set_ylabel('Frequency')
            axes[1,1].set_yscale('log')
            
            plt.tight_layout()
            plt.show()
        
        return {
            'residual': residual,
            'rms_time': residual_rms,
            'max_time': residual_max,
            'global_rms': residual_global_rms
        }
    
    def check_solution_stability(self, traj_idx: int = 0, plot: bool = True) -> Dict[str, float]:
        """
        Check for numerical instabilities and solution quality indicators.
        
        Args:
            traj_idx: Index of trajectory to analyze
            plot: Whether to plot stability analysis
            
        Returns:
            Dictionary containing stability metrics
        """
        data = self.dataset[traj_idx]
        u = data['u_sequence'].numpy()
        
        # Compute derivatives
        u_x, u_xxx = self.compute_spatial_derivatives(u)
        u_t = self.compute_temporal_derivative(u)
        
        # Stability indicators
        max_u = np.max(np.abs(u))
        max_u_x = np.max(np.abs(u_x))
        max_u_t = np.max(np.abs(u_t))
        max_u_xxx = np.max(np.abs(u_xxx))
        
        # Check for exponential growth
        u_norm_time = np.array([np.linalg.norm(u[t]) for t in range(self.n_timesteps)])
        growth_rate = (u_norm_time[-1] - u_norm_time[0]) / (self.time_points[-1] - self.time_points[0])
        
        # Check for oscillations (high frequency content)
        # Look at second differences in time
        if self.n_timesteps > 2:
            u_tt = np.zeros_like(u)
            for t in range(1, self.n_timesteps-1):
                u_tt[t] = (u[t+1] - 2*u[t] + u[t-1]) / self.dt**2
            max_u_tt = np.max(np.abs(u_tt))
        else:
            max_u_tt = 0
        
        # Smoothness indicator - check for spurious oscillations
        smoothness = np.mean(np.abs(u_xxx) / (np.abs(u) + 1e-10))
        
        if plot:
            fig, axes = plt.subplots(2, 2, figsize=(12, 8))
            
            # Solution norm over time
            axes[0,0].plot(self.time_points, u_norm_time, 'b-', linewidth=2)
            axes[0,0].set_title(f'Solution Norm (Growth Rate: {growth_rate:.2e})')
            axes[0,0].set_xlabel('Time')
            axes[0,0].set_ylabel('||u||')
            axes[0,0].grid(True)
            
            # Derivative magnitudes over time
            max_derivs_time = np.array([np.max(np.abs(u_x[t])) for t in range(self.n_timesteps)])
            axes[0,1].semilogy(self.time_points, max_derivs_time, 'r-', linewidth=2)
            axes[0,1].set_title('Maximum |u_x| over Time')
            axes[0,1].set_xlabel('Time')
            axes[0,1].set_ylabel('Max |u_x|')
            axes[0,1].grid(True)
            
            # Check for blow-up or shock formation
            if self.n_timesteps > 10:
                # Look at solution at different times
                times_to_plot = [0, self.n_timesteps//4, self.n_timesteps//2, -1]
                for i, t_idx in enumerate(times_to_plot):
                    if t_idx == -1:
                        t_idx = self.n_timesteps - 1
                    axes[1,0].plot(self.x, u[t_idx], label=f't={self.time_points[t_idx]:.2f}')
                
                axes[1,0].set_title('Solution Profiles')
                axes[1,0].set_xlabel('x')
                axes[1,0].set_ylabel('u')
                axes[1,0].legend()
                axes[1,0].grid(True)
            
            # Residual magnitude over time
            residual = u_t + u * u_x + u_xxx
            residual_norm = np.array([np.linalg.norm(residual[t]) for t in range(self.n_timesteps)])
            axes[1,1].semilogy(self.time_points, residual_norm, 'g-', linewidth=2)
            axes[1,1].set_title('PDE Residual Norm')
            axes[1,1].set_xlabel('Time')
            axes[1,1].set_ylabel('||Residual||')
            axes[1,1].grid(True)
            
            plt.tight_layout()
            plt.show()
        
        return {
            'max_u': max_u,
            'max_u_x': max_u_x,
            'max_u_t': max_u_t,
            'max_u_xxx': max_u_xxx,
            'max_u_tt': max_u_tt,
            'growth_rate': growth_rate,
            'smoothness': smoothness,
            'final_norm': u_norm_time[-1],
            'norm_growth_factor': u_norm_time[-1] / u_norm_time[0] if u_norm_time[0] > 0 else np.inf
        }
    
    def analyze_solution_properties(self, traj_idx: int = 0, plot: bool = True) -> Dict[str, np.ndarray]:
        """
        Analyze general properties of KdV solutions (not assuming solitons).
        
        Args:
            traj_idx: Index of trajectory to analyze
            plot: Whether to plot analysis
            
        Returns:
            Dictionary containing solution properties
        """
        data = self.dataset[traj_idx]
        u = data['u_sequence'].numpy()
        
        # Compute derivatives
        u_x, u_xxx = self.compute_spatial_derivatives(u)
        
        # Solution properties over time
        u_max = np.max(u, axis=1)           # Maximum value
        u_min = np.min(u, axis=1)           # Minimum value  
        u_mean = np.mean(u, axis=1)         # Mean value
        u_std = np.std(u, axis=1)           # Standard deviation
        u_range = u_max - u_min             # Range
        
        # Energy-like quantities
        kinetic_energy = np.array([trapz(u[t]**2, dx=self.dx) for t in range(self.n_timesteps)])
        gradient_energy = np.array([trapz(u_x[t]**2, dx=self.dx) for t in range(self.n_timesteps)])
        
        # Check for shock formation or blow-up
        max_gradient = np.max(np.abs(u_x), axis=1)
        
        if plot:
            fig, axes = plt.subplots(2, 3, figsize=(15, 8))
            
            # Solution evolution heatmap
            im1 = axes[0,0].imshow(u.T, aspect='auto', origin='lower',
                                  extent=[self.time_points[0], self.time_points[-1],
                                         self.x[0], self.x[-1]])
            axes[0,0].set_title('Solution u(x,t)')
            axes[0,0].set_xlabel('Time')
            axes[0,0].set_ylabel('Space')
            plt.colorbar(im1, ax=axes[0,0])
            
            # Gradient evolution
            im2 = axes[0,1].imshow(u_x.T, aspect='auto', origin='lower',
                                  extent=[self.time_points[0], self.time_points[-1],
                                         self.x[0], self.x[-1]])
            axes[0,1].set_title('Gradient u_x(x,t)')
            axes[0,1].set_xlabel('Time')
            axes[0,1].set_ylabel('Space')
            plt.colorbar(im2, ax=axes[0,1])
            
            # Statistics over time
            axes[0,2].plot(self.time_points, u_max, 'r-', label='Max', linewidth=2)
            axes[0,2].plot(self.time_points, u_min, 'b-', label='Min', linewidth=2)
            axes[0,2].plot(self.time_points, u_mean, 'g-', label='Mean', linewidth=2)
            axes[0,2].set_title('Solution Statistics')
            axes[0,2].set_xlabel('Time')
            axes[0,2].set_ylabel('u value')
            axes[0,2].legend()
            axes[0,2].grid(True)
            
            # Energy evolution
            axes[1,0].plot(self.time_points, kinetic_energy, 'r-', label='∫u² dx', linewidth=2)
            axes[1,0].plot(self.time_points, gradient_energy, 'b-', label='∫(u_x)² dx', linewidth=2)
            axes[1,0].set_title('Energy Components')
            axes[1,0].set_xlabel('Time')
            axes[1,0].set_ylabel('Energy')
            axes[1,0].legend()
            axes[1,0].grid(True)
            
            # Standard deviation and range
            axes[1,1].plot(self.time_points, u_std, 'g-', label='Std Dev', linewidth=2)
            axes[1,1].plot(self.time_points, u_range, 'orange', label='Range', linewidth=2)
            axes[1,1].set_title('Solution Spread')
            axes[1,1].set_xlabel('Time')
            axes[1,1].set_ylabel('Value')
            axes[1,1].legend()
            axes[1,1].grid(True)
            
            # Maximum gradient (check for steepening)
            axes[1,2].semilogy(self.time_points, max_gradient, 'purple', linewidth=2)
            axes[1,2].set_title('Maximum |u_x|')
            axes[1,2].set_xlabel('Time')
            axes[1,2].set_ylabel('Max |u_x|')
            axes[1,2].grid(True)
            
            plt.tight_layout()
            plt.show()
        
        return {
            'u_max': u_max,
            'u_min': u_min,
            'u_mean': u_mean,
            'u_std': u_std,
            'u_range': u_range,
            'kinetic_energy': kinetic_energy,
            'gradient_energy': gradient_energy,
            'max_gradient': max_gradient
        }
    
    def validate_multiple_trajectories(self, n_trajectories: int = 10) -> Dict[str, List]:
        """
        Validate multiple trajectories and return statistics.
        
        Args:
            n_trajectories: Number of trajectories to validate
            
        Returns:
            Dictionary containing validation statistics across trajectories
        """
        n_trajectories = min(n_trajectories, self.n_samples)
        
        results = {
            'conservation_errors': [],
            'residual_errors': [],
            'trajectory_indices': list(range(n_trajectories))
        }
        
        print(f"Validating {n_trajectories} trajectories...")
        
        for i in range(n_trajectories):
            print(f"Trajectory {i+1}/{n_trajectories}", end='\r')
            
            # Check conservation laws
            conservation = self.check_conservation_laws(i, plot=False)
            max_I1_error = np.max(conservation['I1_error'])
            max_I2_error = np.max(conservation['I2_error']) 
            max_I3_error = np.max(conservation['I3_error'])
            
            results['conservation_errors'].append({
                'I1': max_I1_error,
                'I2': max_I2_error,
                'I3': max_I3_error
            })
            
            # Check PDE residual
            residual = self.check_pde_residual(i, plot=False)
            results['residual_errors'].append(residual['global_rms'])
        
        print("\nValidation complete!")
        
        # Compute statistics
        conservation_stats = {
            'I1': {
                'mean': np.mean([r['I1'] for r in results['conservation_errors']]),
                'std': np.std([r['I1'] for r in results['conservation_errors']]),
                'max': np.max([r['I1'] for r in results['conservation_errors']])
            },
            'I2': {
                'mean': np.mean([r['I2'] for r in results['conservation_errors']]),
                'std': np.std([r['I2'] for r in results['conservation_errors']]),
                'max': np.max([r['I2'] for r in results['conservation_errors']])
            },
            'I3': {
                'mean': np.mean([r['I3'] for r in results['conservation_errors']]),
                'std': np.std([r['I3'] for r in results['conservation_errors']]),
                'max': np.max([r['I3'] for r in results['conservation_errors']])
            }
        }
        
        residual_stats = {
            'mean': np.mean(results['residual_errors']),
            'std': np.std(results['residual_errors']),
            'max': np.max(results['residual_errors'])
        }
        
        results['conservation_statistics'] = conservation_stats
        results['residual_statistics'] = residual_stats

        return results
        
    def check_single_trajectory_detailed(self, traj_idx: int = 0):
        """
        Detailed analysis of a single trajectory with all plots.
        
        Args:
            traj_idx: Index of trajectory to analyze
            
        Returns:
            Tuple of all analysis results
        """
        print(f"Analyzing trajectory {traj_idx}...")
        
        # Get trajectory info
        data = self.dataset[traj_idx]
        param = data['parameter'].item()
        print(f"Parameter value: {param:.4f}")
        
        # Run all checks with plots
        print("\nKdV Invariants:")
        conservation = self.check_conservation_laws(traj_idx, plot=True)
        
        print("\nPDE residual:")
        residual = self.check_pde_residual(traj_idx, plot=True)
        
        print("\nSolution properties:")
        properties = self.analyze_solution_properties(traj_idx, plot=True)
        
        print("\nSolution stability:")
        stability = self.check_solution_stability(traj_idx, plot=True)
        
        return conservation, residual, properties, stability

    def quick_validation_check(self, n_samples: int = 5) -> bool:
        """
        Quick validation check returning boolean result.
        
        Args:
            n_samples: Number of trajectories to validate
            
        Returns:
            Boolean indicating if validation passed
        """
        try:
            results = self.validate_multiple_trajectories(n_samples)
            
            # Check if results is None or has errors
            if results is None:
                print("Validation failed: No results returned")
                return False
                
            if 'error_occurred' in results and results['error_occurred']:
                print(f"Validation failed: {results.get('error_message', 'Unknown error')}")
                return False
            
            # Check if we have the required statistics
            if 'conservation_statistics' not in results or results['conservation_statistics'] is None:
                print("Validation failed: No conservation statistics available")
                return False
                
            if 'residual_statistics' not in results or results['residual_statistics'] is None:
                print("Validation failed: No residual statistics available")
                return False
            
            # Define thresholds
            conservation_threshold = 1e-3
            residual_threshold = 1e-2
            
            # Check conservation
            conservation_ok = all(
                results['conservation_statistics'][q]['max'] < conservation_threshold 
                for q in ['I1', 'I2', 'I3']
                if q in results['conservation_statistics']
            )
            
            # Check residual
            residual_ok = results['residual_statistics']['max'] < residual_threshold
            
            overall_ok = conservation_ok and residual_ok
            
            if not overall_ok:
                print("Validation failed:")
                if not conservation_ok:
                    print("- KdV invariant violations detected")
                    for q in ['I1', 'I2', 'I3']:
                        if q in results['conservation_statistics']:
                            max_err = results['conservation_statistics'][q]['max']
                            if max_err >= conservation_threshold:
                                print(f"  - {q}: {max_err:.2e} >= {conservation_threshold:.0e}")
                if not residual_ok:
                    print("- High PDE residual detected")
                    print(f"  - Residual: {results['residual_statistics']['max']:.2e} >= {residual_threshold:.0e}")
            else:
                print("✓ Quick validation passed")
            
            return overall_ok
            
        except Exception as e:
            print(f"Validation failed with exception: {e}")
            import traceback
            traceback.print_exc()
            return False
    
    def print_validation_report(self, results: Dict):
        """Print a formatted validation report."""
        if results is None:
            print("Error: No validation results to report")
            return
            
        print("="*60)
        print("KdV EQUATION VALIDATION REPORT")
        print("="*60)
        
        if 'conservation_statistics' in results and results['conservation_statistics'] is not None:
            print("\nKdV INVARIANT CONSERVATION ERRORS:")
            print("-"*30)
            for invariant in ['I1', 'I2', 'I3']:
                if invariant in results['conservation_statistics']:
                    stats = results['conservation_statistics'][invariant]
                    print(f"{invariant:>4}: Mean={stats['mean']:.6e}, Std={stats['std']:.6e}, Max={stats['max']:.6e}")
                else:
                    print(f"{invariant:>4}: No data available")
        else:
            print("\nKdV INVARIANT CONSERVATION ERRORS: No data available")
        
        if 'residual_statistics' in results and results['residual_statistics'] is not None:
            print("\nPDE RESIDUAL ERRORS:")
            print("-"*30)
            stats = results['residual_statistics']
            print(f"{'RESIDUAL':>10}: Mean={stats['mean']:.6e}, Std={stats['std']:.6e}, Max={stats['max']:.6e}")
        else:
            print("\nPDE RESIDUAL ERRORS: No data available")
        
        if 'conservation_statistics' in results and 'residual_statistics' in results:
            print("\nVALIDATION CRITERIA:")
            print("-"*30)
            # Define thresholds (adjust based on your requirements)
            conservation_threshold = 1e-3
            residual_threshold = 1e-2
            
            try:
                conservation_ok = all(
                    results['conservation_statistics'][q]['max'] < conservation_threshold 
                    for q in ['I1', 'I2', 'I3']
                    if q in results['conservation_statistics']
                )
                residual_ok = results['residual_statistics']['max'] < residual_threshold
                
                print(f"KdV Invariants:   {'PASS' if conservation_ok else 'FAIL'} (threshold: {conservation_threshold:.0e})")
                print(f"PDE Residual:     {'PASS' if residual_ok else 'FAIL'} (threshold: {residual_threshold:.0e})")
                print(f"Overall:          {'PASS' if (conservation_ok and residual_ok) else 'FAIL'}")
            except Exception as e:
                print(f"Error computing validation criteria: {e}")
        else:
            print("\nVALIDATION CRITERIA: Insufficient data for assessment")

# Example usage function
def validate_kdv_dataset(dataset, domain_length: float = None, n_trajectories: int = 5):
    """
    Complete validation pipeline for KdV dataset (non-soliton solutions).
    
    Args:
        dataset: TrajectoryDataset instance
        domain_length: Length of spatial domain (e.g., 16.0)
        n_trajectories: Number of trajectories to validate in detail
    """
    validator = KdVValidator(dataset, domain_length=domain_length)
    
    print("Starting KdV validation for general solutions...")
    
    # Detailed analysis of first trajectory
    print("\n1. Checking KdV invariant conservation for first trajectory...")
    conservation_results = validator.check_conservation_laws(traj_idx=0, plot=True)
    
    print("\n2. Checking PDE residual for first trajectory...")
    residual_results = validator.check_pde_residual(traj_idx=0, plot=True)
    
    print("\n3. Analyzing solution properties...")
    properties_results = validator.analyze_solution_properties(traj_idx=0, plot=True)
    
    print("\n4. Checking solution stability...")
    stability_results = validator.check_solution_stability(traj_idx=0, plot=True)
    
    print(f"\n5. Validating {n_trajectories} trajectories...")
    validation_results = validator.validate_multiple_trajectories(n_trajectories)
    
    print(f"\nDEBUG: validate_multiple_trajectories returned: {type(validation_results)}")
    print(f"DEBUG: validation_results is None: {validation_results is None}")
    
    print("\n6. Validation Report:")
    if validation_results is not None:
        validator.print_validation_report(validation_results)
    else:
        print("Error: validate_multiple_trajectories returned None")
        print("Creating minimal report from individual trajectory results...")
        
        # Create a minimal report from what we do have
        print("="*60)
        print("KdV EQUATION VALIDATION REPORT (SINGLE TRAJECTORY)")
        print("="*60)
        
        print(f"\nSingle Trajectory Conservation Errors:")
        print(f"I1 error: {np.max(conservation_results['I1_error']):.6e}")
        print(f"I2 error: {np.max(conservation_results['I2_error']):.6e}")
        print(f"I3 error: {np.max(conservation_results['I3_error']):.6e}")
        
        print(f"\nSingle Trajectory PDE Residual:")
        print(f"Global RMS residual: {residual_results['global_rms']:.6e}")
    
    return validator, validation_results

In [ ]:
# Load your dataset
dataset = TrajectoryDataset("/mnt/home/lserrano/LPSDA/data/OP_ED_train_1024.h5", "dispersion", "train")

In [ ]:
# Method 1: Specify domain length explicitly  
validator = KdVValidator(dataset, domain_length=16.0)

In [ ]:
# Method 2: Use the complete validation pipeline
validator, results = validate_kdv_dataset(dataset, domain_length=16.0, n_trajectories=1)

In [ ]:
# Run this to identify the root cause
validator, results = test_basic_validation(dataset, domain_length=16.0)

In [ ]:
# conservation_checks.py
import numpy as np
import torch

def spectral_dx(u, dx):
    """Return u_x on a periodic grid using spectral derivative (real output)."""
    N = u.shape[-1]
    k = 2 * np.pi * np.fft.fftfreq(N, d=dx)   # angular wavenumbers
    u_hat = np.fft.fft(u)
    ux = np.fft.ifft(1j * k * u_hat).real
    return ux

def compute_invariants(u_traj, dx):
    """
    u_traj: (n_timesteps, n_spatial) numpy array
    dx: spatial spacing
    Returns mass, l2, energy arrays of length n_timesteps
    """
    n_timesteps, n_spatial = u_traj.shape
    mass = np.empty(n_timesteps, dtype=float)
    l2   = np.empty(n_timesteps, dtype=float)
    energy = np.empty(n_timesteps, dtype=float)

    for t in range(n_timesteps):
        u = u_traj[t]
        mass[t] = dx * np.sum(u)                     # correct discrete integral on periodic grid
        l2[t]   = dx * np.sum(u**2)
        ux = spectral_dx(u, dx)
        energy[t] = dx * np.sum(0.5 * ux**2 - (1.0/6.0) * u**3)

    return mass, l2, energy

def drift_metrics(arr, use_relative_if_possible=True, eps=1e-12):
    """
    Compute absolute and relative drift measures.
    - abs_drift = max - min
    - rel_drift = (max-min) / |ref| where ref is arr[0] if |arr[0]|>eps else arr.mean()
    """
    amax = arr.max()
    amin = arr.min()
    abs_drift = amax - amin

    if not use_relative_if_possible:
        return abs_drift, None

    # choose stable reference: prefer arr[0], else mean, else std fallback
    if abs(arr[0]) > eps:
        ref = arr[0]
    elif abs(arr.mean()) > eps:
        ref = arr.mean()
    elif np.std(arr) > eps:
        ref = np.std(arr)
    else:
        ref = eps

    rel_drift = abs_drift / (abs(ref) + eps)
    return abs_drift, rel_drift

def check_dataset_conservation(hdf5_file,
                               operator_type="dispersion",
                               split="train",
                               L=16.0,
                               mass_rel_tol=1e-6,
                               mass_abs_tol=1e-8,
                               l2_rel_tol=1e-3,
                               energy_rel_tol=1e-3,
                               max_trajectories=None):
    """
    Run conservation checks across trajectories and report failures.

    mass_rel_tol: acceptable relative drift for mass when mass is not near zero
    mass_abs_tol: acceptable absolute drift for mass when mass is near zero
    l2_rel_tol, energy_rel_tol: acceptable relative drifts for L2 and energy (w.r.t. initial value)
    """
    ds = TrajectoryDataset(hdf5_file, operator_type=operator_type, split=split)
    dx = L / ds.n_spatial
    n_to_check = len(ds) if max_trajectories is None else min(max_trajectories, len(ds))

    failures = []
    for idx in range(n_to_check):
        item = ds[idx]
        u_seq = item["u_sequence"].numpy()  # (n_timesteps, n_spatial)
        mass, l2, energy = compute_invariants(u_seq, dx)

        mass_abs, mass_rel = drift_metrics(mass)
        l2_abs, l2_rel = drift_metrics(l2)
        e_abs, e_rel = drift_metrics(energy)

        # Decide if mass is "near-zero mean" and therefore use absolute tolerance
        mass_is_near_zero = (abs(mass[0]) < 1e-8) or (abs(mass.mean()) < 1e-8)

        mass_ok = (mass_rel is not None and mass_rel <= mass_rel_tol) or (mass_abs <= mass_abs_tol)
        l2_ok = (l2_rel is not None and l2_rel <= l2_rel_tol)
        e_ok  = (e_rel is not None and e_rel <= energy_rel_tol)

        if not (mass_ok and l2_ok and e_ok):
            failures.append({
                "idx": idx,
                "mass": {"abs_drift": mass_abs, "rel_drift": mass_rel, "near_zero": mass_is_near_zero, "initial": mass[0]},
                "l2":   {"abs_drift": l2_abs,   "rel_drift": l2_rel, "initial": l2[0]},
                "energy":{"abs_drift": e_abs,   "rel_drift": e_rel, "initial": energy[0]}
            })

        # Print a compact per-trajectory summary (optional)
        print(f"traj {idx:03d}: mass abs={mass_abs:.3e}, rel={mass_rel if mass_rel is not None else np.nan:.3e} "
              f"(near_zero={mass_is_near_zero}), l2 rel={l2_rel:.3e}, e rel={e_rel:.3e}")

    # Summary
    print("\nSUMMARY")
    print(f"Checked {n_to_check} trajectories, failures: {len(failures)}")
    if failures:
        print("First few failures:")
        for f in failures[:10]:
            print(f" - traj {f['idx']}: mass_abs={f['mass']['abs_drift']:.3e}, mass_rel={f['mass']['rel_drift']}, "
                  f"l2_rel={f['l2']['rel_drift']:.3e}, e_rel={f['energy']['rel_drift']:.3e}")
    return failures


In [ ]:
check_dataset_conservation("/mnt/home/lserrano/LPSDA/data/OP_ED_train_1024.h5")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def compute_invariants(u, L=1.0):
    """
    Compute KdV invariants for a 1D field u(x).
    Assumes periodic BC and equispaced grid.
    L : domain length (default 1.0, change if needed).
    """
    N = u.shape[-1]
    dx = L / N

    # Fourier wavenumbers
    k = 2*np.pi*np.fft.fftfreq(N, d=dx)
    
    # Derivative using FFT
    ux = np.fft.ifft(1j * k * np.fft.fft(u)).real
    
    # Mass
    mass = np.trapz(u, dx=dx)
    # L2 norm
    l2 = np.trapz(u**2, dx=dx)
    # Energy
    energy = np.trapz(ux**2 - 0.5*(u**3), dx=dx)
    return mass, l2, energy

In [ ]:
traj_id=490
u_seq = dataset[traj_id]["u_sequence"].numpy()   # shape: (T, N)
T, N = u_seq.shape

In [ ]:
masses, l2s, energies = [], [], []
for t in range(T):
    m, l2, e = compute_invariants(u_seq[t], L=16.0)  # <-- adjust L if not [0,1]
    masses.append(m)
    l2s.append(l2)
    energies.append(e)

# ---- Plot ----
plt.figure(figsize=(12,4))
plt.subplot(1,3,1)
plt.plot(masses, label="Mass")
plt.xlabel("time step")
plt.ylabel("Mass")
plt.legend()

plt.subplot(1,3,2)
plt.plot(l2s, label="L2 norm")
plt.xlabel("time step")
plt.ylabel("L2")
plt.legend()

plt.subplot(1,3,3)
plt.plot(energies, label="Energy")
plt.xlabel("time step")
plt.ylabel("Energy")
plt.legend()

plt.tight_layout()
plt.show()